Geting start with Gurobi : https://support.gurobi.com/hc/en-us/articles/14799677517585-Getting-Started-with-Gurobi-Optimizer  
Install Gurobi for Python: https://support.gurobi.com/hc/en-us/articles/360044290292-How-do-I-install-Gurobi-for-Python

In [29]:
# check Gurobi version
import gurobipy as gp
print(gp.gurobi.version())
#print(gp.Model().getParamInfo("LicenseID"))

(12, 0, 1)


In [30]:
from gurobipy import GRB
import random
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time

In [ ]:
# Reads a Graph6 format file from given path.
# Returns a list of NetworkX graph objects (no isomorphic graphs) with unique indix for each selected graph
# read_graphs_from_g6(str path)
def read_graphs_from_g6(path):

    #read .g6 file from path
    graphs = nx.read_graph6(path)

    #total number of graphs (with isomorphic graphs)
    number_of_graphs = len(graphs)

    #store non-isomorphic NetworkX graphs in list graphs_unique
    graphs_unique = []
    for i in range(number_of_graphs):
        g = graphs[i]
        keep = True
        if len(graphs_unique) != 0:
            for g2 in graphs_unique:
                if nx.is_isomorphic(g, g2):
                    keep = False
        if keep:
            graphs_unique.append(g)
    number_uniques = len(graphs_unique)

    #if the total number of graphs exceeds 50, randomly pick 50 graphs. 
    if number_uniques > 50:
        selected_graph_indices = random.sample(range(len(graphs_unique)),50)
        selected_graph_indices.sort()
    else:
        selected_graph_indices = list(range(number_uniques))   

    #get graph sample from selected indices
    graph_sample = []
    for i in selected_graph_indices:
        g = graphs[i]
        graph_sample.append(g)
    
    return selected_graph_indices, graph_sample

In [31]:
#Generate a list of graph_num unique (non-isomorphic) random graphs with n nodes.
def generate_unique_graphs(node_num, graph_num):
    unique_graphs = []

    while len(unique_graphs) < graph_num:

      
        
        G = nx.gnp_random_graph(node_num, 0.5)
        
        # Check if the generated graph is isomorphic to any already in our list.
        if not any(nx.is_isomorphic(G, UG) for UG in unique_graphs):
            unique_graphs.append(G)

    graph_indices = list(range(0, graph_num))

    return graph_indices, unique_graphs

In [32]:
#solvs the apglp using gurobi mip with input graph G, initial value a and constant difference d
#returns the assigned edge weights and time used for processing
#gurobi_mip (NetworkX graph object: G, int: a, int: d)
def gurobi_mip (G, a, d):

    print(f"running: a={a}; b={d}")
    
    #list of vertex lables Y
    vertex_labels = []
    n=G.order()
    for i in range(n):
        vertex_labels.append(a + i*d)

    #record processing time
    start_time = time.process_time()

    m = gp.Model()

    #for model description and log information, comment out the following line
    m.Params.OutputFlag = 0
    
    # add binary variables: t[v, w] == 1 if vertex v is assigned label w
    t = m.addVars(G.nodes, vertex_labels, vtype=GRB.BINARY, name="t")
    
    # add variables for edge labels: x[e] for each edge e, with x[e] >= 1 and <=a+(n-2)*d (the second highest vertex label
    x = m.addVars(G.edges, lb=1, ub=a+(n-2)*d, vtype=GRB.INTEGER, name="x")
    
    #each vertex gets exactly one label
    for v in G.nodes:
        m.addConstr(gp.quicksum(t[v, w] for w in vertex_labels) == 1,
                name=f"one_w_for_v{v}")
    
    #each label is assigned to exactly one vertex
    for w in vertex_labels:
        m.addConstr(gp.quicksum(t[v, w] for v in G.nodes) == 1,
                name=f"one_v_for_w{w}")
    
    #for each vertex, the sum of its incident edge labels equals the vertex label.
    for v in G.nodes:
        #incident edges of vertex v.
        incident_edges = [e for e in G.edges if v in e]
        m.addConstr(gp.quicksum(x[e] for e in incident_edges) == gp.quicksum(w * t[v, w] for w in vertex_labels), name=f"balance_v{v}")
    
    #set a dummy objective (minimize 0) since we only require a feasible solution
    m.setObjective(0, GRB.MINIMIZE)
    
    #run the optimizer to solve the model
    m.optimize()

    #check if an optimal solution was found
    if m.status == GRB.OPTIMAL: 

        #store time in microseconds
        elapsed_time = (time.process_time() - start_time)*(10**6)
        
        #check which label is assigned to each vertex and store it in dic node_weights in format {vertex : weight}
        node_weights={}
        for v in G.nodes:
            for w in vertex_labels:
                if t[v, w].X == 1:  
                    node_weights[v] = w
                               
        edge_weights = {e:int(x[e].X) for e in G.edges}
        
        #draw the graph using NetworkX drawing functions
        pos = nx.spring_layout(G,seed=2)
        plt.figure(figsize=(6, 6))
        nx.draw(G, pos, labels=node_weights, with_labels=True, node_size=500, font_size=12, node_color='white', edgecolors='black', linewidths=3, width=3)
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weights, font_size=10)
        plt.show()
        
        return node_weights, edge_weights, m.Runtime*10**6, m.NumVars, m.NumConstrs, m.NumNZs
    return 0, 0, 0, 0, 0, 0


    

In [33]:
# Iterates over 'a' (from a_min to a_max) and 'd' (from d_min to d_max) for each NetworkX graph in graphs.
# For each parameter combination, compute the APG using the pythonmip function.
# If a valid result is obtained, append it to the provided DataFrame.
# Finally, return the updated DataFrame.
#find_apgs(int a_min, int a_max, int d_min, int d_max, list[NetworkX graph objects] graphs, DataFrame df)
def find_apgs (a_min, a_max, d_min, d_max, graph_indices, graphs, df):

    #extract the column names
    col_names = df.columns.tolist()
    
    total = len(graphs)
    graph_count = 0
    
    #iterates over as and ds for each graph and appends results to df
    for G in graphs:

        graph_index = graph_indices[graph_count]
        
        graph_count += 1
        print(f"Running graph:{graph_count}/{total}")
        
        #adjacency matrix of graph G as a nested list
        ad_list = nx.adjacency_matrix(G, weight=None).toarray().tolist()
        
        for a in range(a_min, a_max):
            for d in range(d_min, d_max):
                vertex_labels, edge_labels, gurobi_runtime, num_vars, num_constrs, nonzero_coefs = gurobi_mip(G, a, d)
                if vertex_labels != 0:
                    df2 = {col_names[0]: graph_index, col_names[1]: G, col_names[2]: ad_list, col_names[3]: a, col_names[4]: d, col_names[5]: "Y",
                    col_names[6]: vertex_labels, col_names[7]: edge_labels, col_names[8]: gurobi_runtime, col_names[9]: num_vars, col_names[10]: num_constrs, col_names[11]: nonzero_coefs}
                    df = pd.concat([df, pd.DataFrame([df2])], ignore_index=True)
    return df

In [34]:
#Takes a .g6 file path and the output file path as inputs, finds the APGs, and writes the results to the output file
#mip_find_apgs(str graph_file_name, str df_file_name)
def mip_find_apgs(node_num, graph_num, df_file_name):

    #define the column names for a DataFrame df
    col_name = ['graph_index', 'Graph','Adjacency_list','a','d','APG_label_availability','Vertex_labels', 'Edge_labels', 'Gurobi_runtime/microseconds', 'Gurobi_num_vars', 'Gurobi_num_constrs', 'Gurobi_nonzero_coefs']
    dict = {col_name[0]:[],
            col_name[1]:[],
            col_name[2]:[],
            col_name[3]:[],
            col_name[4]:[],
            col_name[5]:[],
            col_name[6]:[],
            col_name[7]:[],
            col_name[8]:[],
            col_name[9]:[],
            col_name[10]:[],
            col_name[11]:[],
           }
    #create the DataFrame df with column names defined
    df = pd.DataFrame(dict)

    #set the lower and upper bounds for initial value a and constant difference d
    a_min = 2
    a_max = 30
    d_min = 1
    d_max = 10

    if node_num < 6:
        graph_indices, graphs = read_graphs_from_g6(graph_file_name)
    else:
        graph_indices, graphs = generate_unique_graphs(node_num,graph_num)
    df = find_apgs(a_min, a_max, d_min, d_max, graph_indices, graphs, df)
    
    #randomly select 100 APG samples and write to the output csv file 
    if len(df) >= 100:
        final_df = df.sample(100).sort_index()
    else:
        raise ValueError("Not enough graph samples, at least 100 are required. Try run again.")
    final_df.index.name = 'index'
    final_df.to_csv(df_file_name, index=False)
    

In [ ]:
#From 27: a -> (2,10); From 42: a -> (2,15)； From 57: a -> (2,20); From 70: a -> (2,30)

for i in range(4,60):
    mip_find_apgs(i, 100, f'apgs/apg{i}.csv')